In [ ]:
!pip install -q transformers torch sentencepiece

In [ ]:
from  transformers import AutoTokenizer , AutoModelForSeq2SeqLM
import torch

In [ ]:
model_name = "facebook/bart-large-cnn"

In [ ]:
tokenizer =AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print("Using device:", device)

Using device: cuda


In [ ]:
text = """
Transformers are deep learning models that rely heavily on the attention mechanism.
They became highly influential in natural language processing because they can process
tokens in parallel and capture long-range dependencies more effectively than many older
recurrent neural networks. BART is an encoder-decoder transformer model that is widely
used for text generation tasks such as summarization. It reads the input text, builds
a contextual representation, and then generates a shorter version that preserves the
most important information.
"""

In [ ]:
inputs = tokenizer(
    text,
    max_length=1024,
    truncation=True,
    return_tensors="pt"
)
# out put is input_ids and  attention_mask

In [ ]:
inputs = {key: value.to(device) for key, value in inputs.items()}

In [ ]:
summary_ids= model.generate(
    inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_new_tokens=60,
    num_beams=4,
    early_stopping=True
)


In [ ]:
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print("Summary:")
print(summary)

Summary:
Transformers are deep learning models that rely heavily on the attention mechanism. BART is an encoder-decoder transformer model that is widelyused for text generation tasks such as summarization. It reads the input text, builds a contextual representation, and then generates a shorter version that preserves the most


In [ ]:
print("Original Text:\n")
print(text)

print("\n" + "="*50 + "\n")

print("Generated Summary:\n")
print(summary)

Original Text:


Transformers are deep learning models that rely heavily on the attention mechanism.
They became highly influential in natural language processing because they can process
tokens in parallel and capture long-range dependencies more effectively than many older
recurrent neural networks. BART is an encoder-decoder transformer model that is widely
used for text generation tasks such as summarization. It reads the input text, builds
a contextual representation, and then generates a shorter version that preserves the
most important information.



Generated Summary:

Transformers are deep learning models that rely heavily on the attention mechanism. BART is an encoder-decoder transformer model that is widelyused for text generation tasks such as summarization. It reads the input text, builds a contextual representation, and then generates a shorter version that preserves the most


In [ ]:
def summarize_text(text, max_input_length=1024, max_new_tokens=60, num_beams=4):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        max_length=max_input_length,
        truncation=True
    )

    inputs = {key: value.to(device) for key, value in inputs.items()}

    summary_ids = model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_new_tokens,
        num_beams=num_beams,
        early_stopping=True
    )

    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

In [ ]:
import json
from google.colab import _message

# نحصل على محتوى النوتبوك الحالي
nb = _message.blocking_request("get_ipynb")["ipynb"]

# نحذف metadata الخاصة بال widgets لو موجودة
if "widgets" in nb["metadata"]:
    del nb["metadata"]["widgets"]

# نحفظ نسخة نظيفة
with open("bart_text_summarization_clean.ipynb", "w", encoding="utf-8") as f:
    json.dump(nb, f)

print("Saved clean notebook")